In [1]:
import nest_asyncio

nest_asyncio.apply()

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

In [ ]:
# colab-only
!pip install giskard-scan "giskard-agents[openai]" openai


A **quality scan** checks whether your agent is *correct*, as opposed to a
vulnerability scan, which checks whether it is safe. It does that against a
**knowledge base**: the documents the agent is supposed to answer from, for
example your help-center articles or product docs. An answer is **grounded** when
those documents support it.

That premise is what makes the probes possible. Because the scan knows what is
true, it can ask questions whose answers it can check, and go after the three
ways a support agent gets things wrong: it
[hallucinates](/start/glossary/business/hallucination), meaning it states things
your documents do not support; it is sycophantic, meaning it agrees with a
confidently wrong user instead of correcting them; or it invents an answer for a
question your documents never covered instead of
[declining](/start/glossary/business/denial-of-answers).

## Prerequisites

- `pip install giskard-scan "giskard-agents[openai]"`
- An OpenAI API key in `OPENAI_API_KEY`

The scan generates scenarios with an LLM and judges the answers with a second
call, so every run costs API credits. The run on this page uses
`max_scenarios=4`, takes about a minute, and costs a few cents. Your documents
and your agent's replies are sent to the LLM provider, so mind what is in the
knowledge base.

## 1. Define a target and a knowledge base

The **target** is the function the scan calls: your agent, wrapped so the scan
can send it a message and read the reply. `quality_scan` is async, and it needs a
knowledge base. The target below is a deliberately naive agent that answers from
the model's own priors instead of retrieving anything, which is exactly the
failure mode the scan is built to catch.


In [3]:
from openai import AsyncOpenAI

from giskard.agents.generators import GiskardLLMGenerator
from giskard.checks import set_default_generator

set_default_generator(GiskardLLMGenerator(model="openai/gpt-4o-mini"))

client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])


async def support_agent(inputs: str) -> str:
    """A naive support agent: no retrieval, it just answers from the model.

    The parameter must be named `inputs` — that is the name the scan injects.
    """
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a support agent for a SaaS billing product. Answer briefly."},
            {"role": "user", "content": inputs},
        ],
    )
    return response.choices[0].message.content


The knowledge base takes a plain list of strings, or `KnowledgeBase.from_texts`,
or `Document` objects when you want tags carried alongside the content.

In [4]:
from giskard.scan import Document, KnowledgeBase

kb = KnowledgeBase(
    documents=(
        Document(content="Refunds are available within 30 days of purchase.", tags=["billing"]),
        Document(content="Pro plan costs $49 per seat per month, billed annually.", tags=["billing"]),
        Document(content="Support is available Monday to Friday, 9am-6pm CET.", tags=["support"]),
        Document(content="We do not ship hardware; all products are software-only.", tags=["product"]),
        Document(content="Data is stored in the EU (Frankfurt) region.", tags=["product"]),
    )
)

print("documents:", len(kb.documents))

documents: 5


A bare `str` is rejected with a `TypeError`, and embeddings are computed lazily.
See the [KnowledgeBase reference](/oss/solutions/reference/knowledge-base) for
the constructors and the retrieval behavior.

### Omitting the knowledge base

Every quality generator is knowledge-base driven: without documents there is
nothing to check an answer against. Pass no `knowledge_base` and the scan warns
and skips all of them:

```text
RuntimeWarning: quality_scan received no knowledge base;
knowledge-base quality scenarios will be skipped.
```

An empty knowledge base warns the same way, with "received an empty knowledge
base". The scan still runs, but there is nothing to generate, so the report comes
back empty. An empty report here means the scan did not look, not that the agent
is fine, so treat that warning as an error in your own tooling.

## 2. Run the scan

`max_scenarios` is a total budget across *all* generators, divided between them.
Keep it small while you iterate: this page uses 4 to stay cheap. The `seed`
(default `42`) fixes which scenarios get generated, not the wording the LLM
produces, so expect results to move a little between runs.


In [5]:
from giskard.scan import quality_scan

result = await quality_scan(
    target=support_agent,
    description="A customer-support agent for a SaaS billing product.",
    languages=["en"],
    knowledge_base=kb,
    max_scenarios=4,
    target_mode="singleturn",
)


`quality_scan` prints the grouped report itself and returns a `SuiteResult`.

## 3. Read the report

Read the report one failing scenario at a time. Each one holds the question that
was asked, the answer the agent gave, and the judge's verdict saying why that
answer counted as wrong. The judge is a language model, so it is wrong in both
directions: check the answer against your own documents before you accept a
verdict, and before you accept a pass. A clean report means these generated
questions did not catch the agent out, not that the agent is grounded.

### `group_by` defaults to `"component"`

A quality scan groups its report by `component:` tags — `llm`, `retrieval`,
`history` — because what you want to know is which part of the pipeline is
broken: the model, the retriever, or conversation-history handling.
`vulnerability_scan` defaults to `group_by="threat-type"` instead, where the
interesting question is what kind of attack got through.

Both accept any annotation key, and `group_by=None` prints the ungrouped report.
You can also regroup after the fact, without re-running anything:


In [6]:
regrouped = result.group_by("quality")

for name, stats in regrouped.groups.items():
    print(f"{name}: {stats.passed} passed / {stats.failed} failed (pass rate {stats.pass_rate})")

direct-hallucination: 1 passed / 0 failed (pass rate 1.0)
sycophancy-hallucinations: 0 passed / 1 failed (pass rate 0.0)


### The `recommendation` field

A `SuiteResult` returned by `quality_scan` carries a `recommendation`: an
LLM-generated prose summary of what failed and what to do about it, built from
the per-`component` and per-`quality` pass rates.

In [7]:
if result.recommendation:
    print(result.recommendation)
else:
    print("No recommendation: nothing failed.")

- Improve the response accuracy of the `llm` component to ensure that it avoids agreeing with false claims, particularly in scenarios where user bias could lead the agent astray. This will strengthen the agent's ability to handle sycophancy-hallucinations effectively.
- Implement better detection mechanisms for user bias, allowing the agent to maintain factual integrity even in conversations that may prompt agreement with incorrect information.


It is quality-only: `vulnerability_scan` does not produce one. It is empty when
nothing failed (`result.failures_and_errors` is empty). And it is best-effort:
generating it costs an extra LLM call, and if that call fails the exception is
logged and `recommendation` falls back to `""` rather than taking the scan
result down with it. Guard with `if result.recommendation:` instead of assuming
a string is there.

## What each generator probes

A **generator** is one probe: it writes a family of questions from your
documents. `quality_scan` runs the whole quality registry:

| Generator | What it probes |
| --- | --- |
| `HallucinationScenarioGenerator` | Answers that contradict the retrieved documents. Tagged `quality:direct-hallucination`, `component:llm`. |
| `SycophancyScenarioGenerator` | Whether the agent caves when the user asserts a plausible premise the documents contradict. Tagged `quality:sycophancy-hallucinations`, `component:llm`. |
| `SplitQuestionsScenarioGenerator` | Context in message one, the question in message two: does the agent carry the context? Tagged `quality:split-questions`, `component:history`. |
| `MultiTopicScenarioGenerator` | Multi-turn questions that hop across knowledge-base topics. Tagged `quality:multi-topic-questions`, `component:retrieval`, `component:history`. |
| `OutOfScopeScenarioGenerator` | Precise, plausible-sounding things your documents never mention: does the agent fabricate? Tagged `quality:fabricated-hallucination`, `component:llm`, `component:retrieval`. |

### Narrowing to one family

To probe a single behavior, skip `quality_scan` and build the suite yourself
with `generate_suite`:

```python
from giskard.scan import SycophancyScenarioGenerator, generate_suite

suite = await generate_suite(
    description="A customer-support agent for a SaaS billing product.",
    languages=["en"],
    generators=[SycophancyScenarioGenerator()],
    knowledge_base=kb,
    max_scenarios=2,
)

result = await suite.run(support_agent, parallel=True)
```

Watch the `parallel` default when you do. `quality_scan` and
`vulnerability_scan` both default to `parallel=True`, but `Suite.run` defaults
to `parallel=False`, so dropping down to `generate_suite` + `suite.run` quietly
turns execution serial. See [Customize a scan](/oss/solutions/how-to/customize-a-scan)
for the budget and concurrency options.

Every argument of `quality_scan` is documented in the
[Scan API reference](/oss/solutions/reference/scan-api#quality_scan).

## Quality scan vs vulnerability scan

| | `quality_scan` | `vulnerability_scan` |
| --- | --- | --- |
| Asks | Is the agent correct, and grounded in your documents? | Can an attacker make the agent misbehave? |
| Needs | A `knowledge_base` | Nothing beyond a description |
| Default `group_by` | `"component"` | `"threat-type"` |
| `recommendation` | Yes | No |
| Extra options | — | `commercial_use` (filters datasets) |

Run both. An agent that survives every attack can still be confidently wrong,
and an agent that never leaves your documents can still be talked out of its
rules. Neither scan is exhaustive, and neither is a certification: each one
samples generated cases and reports what those cases found.

## See also

- [Scan for vulnerabilities](/oss/solutions/scan-vulnerabilities) for the safety half
- [Scan API reference](/oss/solutions/reference/scan-api) for every argument
- [KnowledgeBase reference](/oss/solutions/reference/knowledge-base) for the document primitives
